In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Data Aggregation").getOrCreate()

26/05/05 12:01:51 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
listings = spark.read.csv("listings.csv.gz", header=True, inferSchema=True, sep=",", quote='"', escape='"', multiLine=True, mode="PERMISSIVE")

In [3]:
reviews = spark.read.csv("reviews.csv.gz", header=True, inferSchema=True, sep=",", quote='"', escape='"', multiLine=True, mode="PERMISSIVE")

In [4]:
# count the number of reviews per listing using the "reviews" dataset
reviews_per_listing = reviews.groupBy('listing_id').count().show()

[Stage 4:>                                                          (0 + 1) / 1]

+----------+-----+
|listing_id|count|
+----------+-----+
|     78606|    2|
|    444886|   12|
|    466017|   28|
|    991477|    5|
|   2557853|   92|
|   2736493|    4|
|   3132302|    3|
|   3734796|    5|
|   3997029|    7|
|   3917692|    1|
|   4361078|   71|
|   5355817|   18|
|   5520243|    6|
|   5921026|   40|
|   6311069|    3|
|   6552071|    1|
|   6651481|   48|
|   6606418|  262|
|   7188835|  206|
|   7709953|   34|
+----------+-----+
only showing top 20 rows


In [6]:
# compute the total number of listings and average review score per host
from pyspark.sql.functions import avg, count

host_stats = listings.filter(listings.review_scores_rating.isNotNull()).groupBy('host_id').agg(count('id').alias('total_listings'), avg('review_scores_rating').alias('average_review_score')).show(10)

[Stage 7:>                                                          (0 + 1) / 1]

+--------+--------------+--------------------+
| host_id|total_listings|average_review_score|
+--------+--------------+--------------------+
| 2038199|             1|                 5.0|
| 2358441|             1|                4.86|
| 2876123|             2|  4.9399999999999995|
| 4157822|             2|  4.9350000000000005|
|  719504|             1|                4.96|
| 7950720|             1|                4.86|
| 6572018|             1|                 5.0|
|12122942|             1|                4.93|
|13851928|             1|                4.97|
|13739634|             2|                4.75|
+--------+--------------+--------------------+
only showing top 10 rows


In [7]:
# find the top ten listings with the highest number of reviews
reviews.groupBy('listing_id').count().orderBy('count', ascending=False).limit(10).show()

[Stage 10:>                                                         (0 + 1) / 1]

+----------+-----+
|listing_id|count|
+----------+-----+
|  47408549| 1902|
|  43120947| 1647|
|  19670926| 1443|
|   2126708| 1142|
|  46233904| 1002|
|   2659707|  998|
|  27833488|  951|
|   4748665|  933|
|  42081759|  914|
|   5266466|  909|
+----------+-----+



In [8]:
# find the top five neighborhoods with the most listings
listings.groupBy('neighbourhood_cleansed').count().orderBy('count', ascending=False).limit(5).show()

[Stage 13:>                                                         (0 + 1) / 1]

+----------------------+-----+
|neighbourhood_cleansed|count|
+----------------------+-----+
|           Westminster|11385|
|         Tower Hamlets| 7469|
|                Camden| 6551|
|  Kensington and Ch...| 6401|
|               Hackney| 6359|
+----------------------+-----+



In [9]:
# get a data frame with the following four columns:
# listings ID,  Listings Name, reviewers name, reviews comment - use "join" to combine data from two datasets
listings.join(reviews, listings.id == reviews.listing_id, 'inner').select(listings.id, 'name', 'reviewer_name', 'comments').show(truncate=False)

[Stage 16:>                                                         (0 + 1) / 1]

+-----+-----------------------------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|id   |name                               |reviewer_name|comments   

In [10]:
# get top five listings with the highest average review comment length. Only return listings with at least 5 reviews. use the "length" function from the "pyspark.sqk.functions" to get a length of a review
from pyspark.sql.functions import length, avg, count

reviews_with_comment_length = reviews.withColumn('comment_length', length('comments'))
reviews_with_comment_length.join(listings, reviews_with_comment_length.listing_id == listings.id, 'inner').groupBy('listing_id').agg(avg(reviews_with_comment_length.comment_length).alias('average_comment_length'), count(reviews_with_comment_length.id).alias('reviews_count')).filter('reviews_count >= 5').orderBy('average_comment_length', ascending=False).show()

[Stage 19:>                                                         (0 + 1) / 1]

+------------------+----------------------+-------------+
|        listing_id|average_comment_length|reviews_count|
+------------------+----------------------+-------------+
|618608352812465378|    1300.1666666666667|            6|
|          28508447|    1089.3333333333333|            6|
|          22661311|     1035.857142857143|            7|
|          53145228|    1006.6666666666666|            6|
|627425975703032358|     951.7777777777778|            9|
|           2197681|                 939.2|            5|
|          13891813|                 905.0|            5|
|            979753|     893.9230769230769|           13|
|630150178279666225|     890.7272727272727|           11|
|           8856894|     890.1666666666666|            6|
|          33310686|     885.8333333333334|            6|
|          22524075|                 885.0|            5|
|          29469389|                 885.0|            6|
|           5555679|     878.7169811320755|          106|
|           65

In [11]:
# using the "join" operator find listings without reviews. Hint: use "left join" of "left anti" join type when implementing this
joined_df = listings.join(reviews, listings.id == reviews.listing_id, how='left_outer')

joined_df.filter(reviews.id.isNull()).select('name').show(truncate=False)

[Stage 23:>                                                         (0 + 1) / 1]

+-------------------------------------------------+
|name                                             |
+-------------------------------------------------+
|ChiqDoube Room in PrivateAppartment              |
|ROOM TO RENT IN THE OLYMPIC PERIOD               |
|Luxury Central London House with Gym and Reformer|
|Stunning Shared Penthouse Apartment              |
|Luxury single room                               |
|Coming to London for the Olympics?               |
|Double bedroom near Olympic Park                 |
|SPARE ROOM TO LET DURING OLYMPICS                |
|Your Studio Flat in London-Olympics              |
|Large Cosy Apartment with Garden E7              |
|Central London flat + skyline views              |
|Trellick Tower, Portobello Rd.                   |
|AROOM TO LET - month                             |
|Cosy 1 bedroom with Thames View                  |
|Single bedroom or double bedroom                 |
|LONDON ALEXANDRA PALACE ROOM £30                 |
|Nice room i